# CSE 532 Final Project
Hello and welcome to **Jackson Kyle** and **Samuel Cooper's** final project for CSE 532.


Research Question 1:
Is there a relationship between metadata about a film and the public perception of the quality of the film.
Judging if a film is good or not is a complicated answer to deliver about any given film, like most art forms, a film's quality is a subjective opinion formed by any viewer or consumer of the film itself. But there may be external factors, that influence whether or not a film will be considered good or bad. Are there external factors that influence whether or not a movie is considered worthwhile or not.
Using the movie data base (TMDB) as the main reference and comparing the average vote for each movie compared and averaged with the rating of famous movie critic Roger Ebert, the question of whether or not meta data about a film has a relationship with the public perception of the quality of a film. 
In this project, we investigate whether metadata can influence public perception of the quality of a movie. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# 1.1) Data Loading:

In [2]:
# Loads in the tmdb datasets
tmdb_movies = pd.read_csv("tmdb_5000_movies.csv")
tmdb_credits = pd.read_csv("tmdb_5000_credits.csv")

# Renames the movie_id column to id for consistency and merging
tmdb_credits.rename(columns={"movie_id":"id"}, inplace=True)
tmdb = pd.merge(tmdb_movies, tmdb_credits, on="id")
# Cleans up duplicate columns created from the merge
tmdb.rename(columns={"title_x":"title"}, inplace=True)
tmdb.drop(columns=["title_y"], inplace=True)

# Loads in the rotten tomatoes datasets with the desired information
rt_movies = pd.read_csv("rotten_tomatoes_movies.csv", usecols=["rotten_tomatoes_link", "movie_title"])
rt_reviews = pd.read_csv("rotten_tomatoes_critic_reviews.csv", usecols=["rotten_tomatoes_link", "critic_name", "review_score"])

# Extracts the Ebert reviews from the dataset
ebert_reviews = rt_reviews.loc[rt_reviews['critic_name'] == "Roger Ebert"].reset_index(drop=True)
ebert_reviews = ebert_reviews.merge(rt_movies[['rotten_tomatoes_link', 'movie_title']],on='rotten_tomatoes_link',how='left')

# Creates the full movie info data frame
movie_info = pd.merge(tmdb, ebert_reviews, left_on="title", right_on="movie_title", how="inner")

# For the sake of our research, we want to add profit and ROI columns:
movie_info["profit"] = movie_info["revenue"] - movie_info["budget"]
#movie_info["roi"] = movie_info["profit"] / movie_info["budget"]

# 1.2) Data Cleaning

In [3]:
# Removes all submissions that have an invalid review
movie_info = movie_info.dropna(subset=["review_score"])
# Homepage and tagline columns are dropped because they don't contain needed information and have multiple null values
movie_info = movie_info.drop(columns=["homepage", "tagline"])

'''Converts the original string rating to a number represented as a percentage'''
def enumerate_rating(score):
    try:
        numerator, denominator = score.split("/")
        return float(numerator) / float(denominator)
    except:
        return None

movie_info["review_score"] = movie_info["review_score"].apply(enumerate_rating)

# Checks to see how many null values are in each row
print(movie_info.isnull().sum())

budget                  0
genres                  0
id                      0
keywords                0
original_language       0
original_title          0
overview                0
popularity              0
production_companies    0
production_countries    0
release_date            0
revenue                 0
runtime                 0
spoken_languages        0
status                  0
title                   0
vote_average            0
vote_count              0
cast                    0
crew                    0
rotten_tomatoes_link    0
critic_name             0
review_score            0
movie_title             0
profit                  0
dtype: int64


For cleaning our data, there were multiple choices we made so that we could make the most out of the data for our research questions. To begin, we dropped the movies that did not contain a review score, as there were less than 10 of them, and would not contribute to the overall data that much. Following this, we dropped the homepage and tagline rows, as they both contained many null values and had no correlation to the purpose of our research. Finally, we wanted the review score to be represented as a number, rather than a string, so that calculations could be done easier. To do this, we created the function enumerate_rating to parse the review_score and convert it to a float value. We then applied this function to each row of the data.

# 1.3) Descriptive Statistics

In [4]:
print("Revenue Statistics:")
print(movie_info["revenue"].describe())
print(f"var      {movie_info["revenue"].var()}\n")

print("Ebert Rating Statistics:")
print(movie_info["review_score"].describe())
print(f"var      {movie_info["review_score"].var()}\n")

print("TMDB Rating Statistics:")
print(movie_info["vote_average"].describe())
print(f"var      {movie_info["vote_average"].var()}")

Revenue Statistics:
count    2.783000e+03
mean     1.012670e+08
std      1.696275e+08
min      0.000000e+00
25%      3.040468e+06
50%      3.837650e+07
75%      1.224673e+08
max      2.787965e+09
Name: revenue, dtype: float64
var      2.8773488125092476e+16

Ebert Rating Statistics:
count    2783.000000
mean        0.700494
std         0.216363
min         0.000000
25%         0.500000
50%         0.750000
75%         0.875000
max         1.000000
Name: review_score, dtype: float64
var      0.046812974713217714

TMDB Rating Statistics:
count    2783.000000
mean        6.371182
std         0.809942
min         2.900000
25%         5.900000
50%         6.400000
75%         6.900000
max         8.500000
Name: vote_average, dtype: float64
var      0.6560060426441428


For our first research question, we're defining a movies success as it's ratings and the revenue it accrued.

Revenue Statistics:
Looking at the statistics obtained from the revenue of each movie, it is easy to tell that there are many outliers, and the revenue values are spread across a large range of values. However, this is to be expected. From the standard deviation, we can see that there is a wide variation in success measured by revenue.

Ebert and TMDB Rating Statistics :
From the statistics obtained from the Ebert and TMDB ratings, we can see that the two yield fairly similar mean values for critical success. Using these two in tandem will be strong in determining the critical success of different films. Additionally, the mean and medians of the two are similar, indicating that there are not many outliers. The standard deviation for each is also fairly low, showing that there isn't too much variation in critical success.

Next, we want to see if there are any correlations between the metadata and the success of the film.

In [7]:
movie_info[["revenue", "vote_average", "review_score", "budget", "popularity", "runtime", "profit"]].corr()

,revenue,vote_average,review_score,budget,popularity,runtime,profit
revenue,1.000000,0.172529,0.078795,0.687666,0.678501,0.235068,0.978394
vote_average,0.172529,1.000000,0.525509,-0.072650,0.388434,0.343086,0.223275
review_score,0.078795,0.525509,1.000000,-0.052402,0.151959,0.236370,0.107445
budget,0.687666,-0.072650,-0.052402,1.000000,0.508391,0.241746,0.522702
popularity,0.678501,0.388434,0.151959,0.508391,1.000000,0.244511,0.651936
runtime,0.235068,0.343086,0.236370,0.241746,0.244511,1.000000,0.207179
profit,0.978394,0.223275,0.107445,0.522702,0.651936,0.207179,1.000000


From this, we want to answer a number of questions about the metadata that correlates to a movie's success:

-**Do higher-rated movies make more money? (Using TMDB and Ebert ratings)**\
-**Do higher budget movies make more money?**\
-**Do different genres correspond to more success?**

In [11]:
movie_info_tmdb_rating = movie_info.groupby(pd.cut(movie_info["vote_average"], bins=10), observed=False)[["revenue", "profit"]].mean()
print(movie_info_tmdb_rating.head(10))

                    revenue        profit
vote_average                             
(2.894, 3.46]  9.024026e+06 -3.547403e+06
(3.46, 4.02]   9.770166e+06 -3.668438e+07
(4.02, 4.58]   3.399311e+07 -4.606887e+06
(4.58, 5.14]   4.373063e+07  8.963321e+06
(5.14, 5.7]    7.235685e+07  3.262705e+07
(5.7, 6.26]    9.285042e+07  5.170156e+07
(6.26, 6.82]   9.975859e+07  6.629733e+07
(6.82, 7.38]   1.159572e+08  8.497178e+07
(7.38, 7.94]   1.630714e+08  1.317083e+08
(7.94, 8.5]    1.979261e+08  1.703600e+08


# 1.4) Data Visualization